In [11]:
import os
import sys
import calendar
import logging
import concurrent.futures
import geopandas as gpd
import xarray as xr
from retry import retry
import earthnet_minicuber as emc
from mypythonlib import myfunctions, phenolopy
import os
import xarray as xr
from tqdm import tqdm
import xarray as xr
import pandas as pd

In [18]:
idx = 8
output_folder= "/Net/Groups/BGI/scratch/fmueller/Data/test_s2/8/"
path= "/Net/Groups/BGI/scratch/fmueller/Data/test_s2/8/8_merged.nc"

In [19]:
import os
print(os.path.exists(path))
print(os.access(path, os.R_OK))
import netCDF4

try:
    dataset = netCDF4.Dataset(path)
    print("File opened successfully with netCDF4.")
except Exception as e:
    print(f"Error opening file with netCDF4: {e}")


import xarray as xr

try:
    dataset = xr.open_dataset(path)
    print("File opened successfully with xarray.")
except Exception as e:
    print(f"Error opening file with xarray: {e}")


dataset = xr.open_dataset(path)
sorted_dataset = dataset.sortby('time')
sorted_dataset

True
True
File opened successfully with netCDF4.
File opened successfully with xarray.


<xarray.Dataset> Size: 56GB
Dimensions:   (time: 474, lat: 1024, lon: 1024)
Coordinates:
    angle     <U6 24B ...
  * time      (time) datetime64[ns] 4kB 2015-08-16 2015-10-05 ... 2023-12-27
  * lon       (lon) float64 8kB -95.03 -95.03 -95.03 ... -94.81 -94.81 -94.81
  * lat       (lat) float64 8kB 34.11 34.11 34.11 34.11 ... 33.93 33.93 33.93
Data variables: (12/15)
    s2_SCL    (time, lat, lon) float64 4GB ...
    s2_mask   (time, lat, lon) float64 4GB ...
    s2_avail  (time) uint8 474B ...
    s2_B01    (time, lat, lon) float64 4GB ...
    s2_B02    (time, lat, lon) float64 4GB ...
    s2_B03    (time, lat, lon) float64 4GB ...
    ...        ...
    s2_B07    (time, lat, lon) float64 4GB ...
    s2_B08    (time, lat, lon) float64 4GB ...
    s2_B8A    (time, lat, lon) float64 4GB ...
    s2_B09    (time, lat, lon) float64 4GB ...
    s2_B11    (time, lat, lon) float64 4GB ...
    s2_B12    (time, lat, lon) float64 4GB ...
Attributes:
    history:  Created on 2024-06-18 17:08:53.153859 with the earthnet-minicub...

In [21]:

# Extract the 'time' dimension
time_values = sorted_dataset['time'].values

# Convert to pandas datetime if not already
time_values = pd.to_datetime(time_values)

# Create a DataFrame to use groupby and count
time_df = pd.DataFrame(time_values, columns=['time'])

# Extract the year from the datetime
time_df['year'] = time_df['time'].dt.year

# Group by the year and count the number of occurrences
yearly_counts = time_df.groupby('year').size()

print(yearly_counts)

year
2015     4
2016    18
2017    29
2018    69
2019    72
2020    73
2021    67
2022    71
2023    71
dtype: int64


In [3]:
# Expected years range
expected_years = range(2015, 2024)
expected_files = [f"{idx}_{year}_{month:02d}.nc" for year in expected_years for month in range(1, 13)]
nc_files = [file for file in os.listdir(output_folder) if file.endswith('.nc')]

# Check if all expected files are present
missing_files = [file for file in expected_files if file not in nc_files]
if missing_files:
    print(f"Missing files: {missing_files}")
    print(f"Breaking off the merging ... ")
else:
    valid_files = []
    empty_files = []

    for file in nc_files:
        file_path = os.path.join(output_folder, file)
        try:
            ds = xr.open_dataset(file_path, chunks={})
            if ds.dims:  # Check if the dataset has any dimensions
                valid_files.append(file)
            else:
                empty_files.append(file)
        except Exception as e:
            print(f"Error opening file {file}: {e}")
            empty_files.append(file)
    
    if empty_files:
        print(f"Empty files: {empty_files}")
        print(f"Removing empty files from the merging list.")

    if valid_files:
        merged_filename = os.path.join(output_folder, f"{idx}_merged.nc")
        print(f"Merging valid NetCDF files into {merged_filename}")

        # Specify coords='minimal' to handle differing coordinates
        file_paths = [os.path.join(output_folder, file) for file in valid_files]
        
        # Use dask for parallel processing and lazy loading
        datasets = [xr.open_dataset(fp, chunks={}) for fp in tqdm(file_paths, desc="Opening files")]
        
        # Perform lazy concatenation
        merged_ds = xr.concat(datasets, dim='time', coords='minimal')

        # Write the merged dataset to disk
        merged_ds.to_netcdf(merged_filename, compute=True)
        print("End merging")
    else:
        print("No valid files found for merging.")



Error opening file 8_merged.nc: [Errno -101] NetCDF: HDF error: '/Net/Groups/BGI/scratch/fmueller/Data/test_s2/8/8_merged.nc'
Empty files: ['8_2015_04.nc', '8_2015_03.nc', '8_merged.nc', '8_2015_07.nc', '8_2015_06.nc', '8_2015_02.nc', '8_2015_05.nc', '8_2015_01.nc', '8_2015_09.nc']
Removing empty files from the merging list.
Merging valid NetCDF files into /Net/Groups/BGI/scratch/fmueller/Data/test_s2/8/8_merged.nc


Opening files: 100%|██████████| 100/100 [00:34<00:00,  2.88it/s]


PermissionError: [Errno 13] Permission denied: '/Net/Groups/BGI/scratch/fmueller/Data/test_s2/8/8_merged.nc'